# CNeuroMod QA — tSNR brain maps

Volumetric montages of average temporal-SNR maps (MNI space), one panel per subject and one per dataset, read directly from the `tsnr` derivative of `source_data/cneuromod.all/{dataset}/`. Figures are written to `output_data/figures/tsnr_maps/`.

tSNR is volumetric and QA cares about signal dropout in ventral/orbitofrontal, temporal and subcortical regions, so we render faithful volumetric slices (nilearn) rather than a cortical surface, which would discard subcortex/cerebellum.

In [1]:
import os
from pathlib import Path

import nibabel as nib
import numpy as np
from nilearn import plotting

# Paths are provided by `invoke run-notebooks` as environment variables.
# Figures go in output_data/figures/{FIG_NAME}/ (also the notebook's "already
# ran" sentinel); the avgtsnr maps this notebook reads are never persisted —
# they live only in the source `tsnr` derivative, fetched by `invoke fetch`.
FIG_NAME = "tsnr_maps"
OUTPUT_DIR = Path(os.environ.get("OUTPUT_DATA_DIR", "../output_data"))
SOURCE_DIR = Path(os.environ.get("SOURCE_DATA_DIR", "../source_data")) / "cneuromod.all"
FIG_DIR = OUTPUT_DIR / "figures" / FIG_NAME
FIG_DIR.mkdir(parents=True, exist_ok=True)

# The upstream per-subject average tSNR map, MNI space (see analysis/tsnr_maps.py).
SPACE = "MNI152NLin2009cAsym"
SUBJECT_AVG_GLOB = f"sub-*/sub-*_space-{SPACE}_stat-avgtsnr_statmap.nii.gz"

# Shared display settings so panels are visually comparable.
CMAP = "inferno"
DISPLAY_MODE = "z"       # axial montage — shows ventral/subcortical dropout
N_CUTS = 8
DPI = 120

In [2]:
from nilearn.image import resample_to_img
from nilearn.plotting import find_cut_slices


def discover_datasets():
    """Dataset names under SOURCE_DIR whose tsnr derivative has >=1 avgtsnr map."""
    datasets = []
    if SOURCE_DIR.is_dir():
        for tsnr_dir in sorted(SOURCE_DIR.glob("*/tsnr")):
            maps = [p for p in tsnr_dir.glob(SUBJECT_AVG_GLOB) if p.is_file()]
            if maps:
                datasets.append(tsnr_dir.parent.name)
    return datasets


def subject_maps(dataset):
    """Sorted per-subject avgtsnr map paths for one dataset, read from source_data."""
    tsnr_dir = SOURCE_DIR / dataset / "tsnr"
    return sorted(p for p in tsnr_dir.glob(SUBJECT_AVG_GLOB) if p.is_file())


def dataset_average_image(dataset):
    """Mean tSNR image over a dataset's subject maps, computed in memory (nothing written).

    Maps from different subjects may sit on slightly different grids, so each is
    resampled to the first subject's grid before averaging. NaNs are ignored
    voxelwise so a subject missing coverage never blanks a voxel for everyone.
    """
    paths = subject_maps(dataset)
    reference = nib.load(str(paths[0]))
    stack = []
    for path in paths:
        image = nib.load(str(path))
        if image.shape != reference.shape or not np.allclose(image.affine, reference.affine):
            image = resample_to_img(image, reference, copy_header=True)
        stack.append(np.asarray(image.dataobj, dtype=np.float32))
    mean = np.nanmean(np.stack(stack, axis=-1), axis=-1)
    return nib.Nifti1Image(mean, reference.affine, reference.header)


def robust_vmax(images):
    """98th percentile of positive tSNR across images — a shared, outlier-robust ceiling."""
    values = []
    for image in images:
        data = np.asarray(image.dataobj, dtype=np.float32)
        data = data[np.isfinite(data) & (data > 0)]
        if data.size:
            values.append(np.percentile(data, 98))
    return float(np.median(values)) if values else None


datasets = discover_datasets()
print(f"found avgtsnr maps for: {datasets}")
if not datasets:
    print("No tSNR maps found — run `invoke fetch` first (needs data access).")
dataset_images = {dataset: dataset_average_image(dataset) for dataset in datasets}
VMAX = robust_vmax(dataset_images.values())
print(f"shared vmax = {VMAX}")

# Cut coordinates are chosen once per dataset from its average map, then reused
# for every subject panel in that dataset — so slices are picked from the
# strongest, least noisy signal (the average) rather than autoselected
# independently per subject, keeping panels anatomically comparable.
dataset_cut_coords = {
    dataset: find_cut_slices(image, direction=DISPLAY_MODE, n_cuts=N_CUTS)
    for dataset, image in dataset_images.items()
}


found avgtsnr maps for: ['floc', 'retinotopy', 'things']


shared vmax = 53.175697326660156


In [3]:
def plot_tsnr(stat_map, title, out_path, cut_coords, vmax=None):
    """Axial montage of one tSNR map (path or in-memory image) on the MNI template."""
    display = plotting.plot_stat_map(
        stat_map, display_mode=DISPLAY_MODE, cut_coords=cut_coords,
        cmap=CMAP, vmax=vmax, colorbar=True, black_bg=True,
        title=title, symmetric_cbar=False,
    )
    display.savefig(str(out_path), dpi=DPI)
    display.close()


# One montage per dataset average (computed in memory, nothing written to disk).
for dataset, image in dataset_images.items():
    plot_tsnr(image, f"{dataset} — average tSNR",
              FIG_DIR / f"{dataset}_avgtsnr.png", dataset_cut_coords[dataset], vmax=VMAX)


/home/pbellec/git/cneuromod.all.qa_figures/.venv/lib/python3.12/site-packages/numpy/lib/_function_base_impl.py:2599: RuntimeWarning: invalid value encountered in <lambda> (vectorized)
  outputs = ufunc(*args, out=...)


In [4]:
# One montage per subject, per dataset — read straight from source_data, sliced
# at the same cut coordinates as that dataset's average for comparability.
for dataset in datasets:
    for path in subject_maps(dataset):
        subject = path.name.split("_", 1)[0]  # e.g. sub-01
        plot_tsnr(str(path), f"{dataset} — {subject} tSNR",
                  FIG_DIR / f"{dataset}_{subject}_avgtsnr.png",
                  dataset_cut_coords[dataset], vmax=VMAX)


/home/pbellec/git/cneuromod.all.qa_figures/.venv/lib/python3.12/site-packages/numpy/lib/_function_base_impl.py:2599: RuntimeWarning: invalid value encountered in <lambda> (vectorized)
  outputs = ufunc(*args, out=...)


/home/pbellec/git/cneuromod.all.qa_figures/.venv/lib/python3.12/site-packages/numpy/lib/_function_base_impl.py:2599: RuntimeWarning: invalid value encountered in <lambda> (vectorized)
  outputs = ufunc(*args, out=...)


/home/pbellec/git/cneuromod.all.qa_figures/.venv/lib/python3.12/site-packages/numpy/lib/_function_base_impl.py:2599: RuntimeWarning: invalid value encountered in <lambda> (vectorized)
  outputs = ufunc(*args, out=...)
